In [ ]:
import json
import os
from typing import Any

import pandas as pd

from kibad_llm.config import PROJ_ROOT

# swith to project root to use same paths as in commands
os.chdir(PROJ_ROOT)

pd.set_option("max_colwidth", 400)

In [ ]:
predictions_file = "data/results/2025-11-25_09-23-55/predictions.jsonl"
ground_truth_file = "data/interim/faktencheck-db/faktencheck-db-converted_2025-11-05.jsonl"

FLATTEN_DICTS = True

In [ ]:
# this below is the exact same code used in the evaluation workflow

import os
from typing import Hashable

from kibad_llm.dataset.json import read_and_preprocess
from kibad_llm.dataset.utils import merge_references_into_predictions


def load_predictions_with_references(
    pred_file: str, ref_file: str
) -> dict[Hashable, dict[str, dict]]:
    predictions = read_and_preprocess(
        file=pred_file,
        id_key="file_name",
        process_id=lambda x: os.path.splitext(x)[0],
        preprocess=lambda x: x.get("structured", None),
    )
    references = read_and_preprocess(
        file=ref_file,
        id_key="zotitem_ptr_id",
    )
    result = merge_references_into_predictions(predictions, references)
    return result


dataset = load_predictions_with_references(predictions_file, ground_truth_file)
f"loaded {len(dataset)} predictions"

In [ ]:
# this is the same code as in the evaluation workflow when using metric.flatten_dicts=true

from kibad_llm.utils.dictionary import flatten_dict_simple

if FLATTEN_DICTS:

    # flatten_dict_simple does not handel nested entries, so we need to call it for prediction and reference individually.
    # Also, prediction may be None for some documents, so we use "{}" in these cases.
    dataset = {
        k: {
            "prediction": flatten_dict_simple(v["prediction"] or {}),
            "reference": flatten_dict_simple(v["reference"]),
        }
        for k, v in dataset.items()
    }

In [ ]:
predictions_df = pd.DataFrame.from_dict({k: v["prediction"] for k, v in dataset.items()}).T
reference_df = pd.DataFrame.from_dict({k: v["reference"] for k, v in dataset.items()}).T

print(f"available columns:\n{sorted(set(reference_df.columns) & set(predictions_df.columns))}")

In [ ]:
def get_column(col: str, replace_nan: Any | None = None) -> pd.DataFrame:
    result = pd.concat({"prediction": predictions_df[col], "reference": reference_df[col]}, axis=1)
    if replace_nan is not None:
        # we can not use .fillna(), since that does not work with lists
        mask = result.isna()
        result[mask] = result[mask].map(lambda _: replace_nan)

    return result


get_column("taxa.german_name", [])

# old version below

In [ ]:
# load JSONL files for ground truth and predictions
with open(predictions_file, "rb") as p_in:
    predictions = [json.loads(line) for line in p_in]

with open(ground_truth_file, "rb") as g_in:
    ground_truth = [json.loads(line) for line in g_in]

In [ ]:
# collect and sort the zotero file ids
predictions_zotero_keys = sorted([p["file_name"][:8] for p in predictions])
predictions = sorted(predictions, key=lambda x: x["file_name"])
# filter ground truth file to retain only entries for which we have predictions
ground_truth_filtered = sorted(
    [g for g in ground_truth if g["zotitem_ptr_id"] in predictions_zotero_keys],
    key=lambda x: x["zotitem_ptr_id"],
)

In [ ]:
# change structure of predictions to 'flatten' it to a list of keys where keys
# are the schema types we extract, plus doc id
predictions_flat = [dict(p["structured"] or {}) for p in predictions]
for idx, p in enumerate(predictions_flat):
    p.update({"id": predictions[idx]["file_name"][:8]})

# change structure of ground truth to retain only predicted schema types plus id
ground_truth_filtered_flat = [
    {k: v for k, v in g.items() if k in predictions_flat[0].keys()} for g in ground_truth_filtered
]
for idx, g in enumerate(ground_truth_filtered_flat):
    g.update({"id": ground_truth_filtered[idx]["zotitem_ptr_id"]})


def flatten(prediction, key_name, flattened_keys):

    if key_name in prediction and prediction[key_name] and len(prediction[key_name]) > 0:
        flattened_lists = [
            [e[k[len(key_name) + 1 :]] for e in prediction[key_name]] for k in flattened_keys
        ]
    else:
        flattened_lists = [[] for _ in flattened_keys]
    return flattened_lists


# change structure of nested / compound types to have flat lists per subtype
compounds = {
    "taxa": ["taxa.german_name", "taxa.scientific_name", "taxa.species_group"],
    "direct_driver": ["direct_driver.category", "direct_driver.details"],
    "indirect_driver": ["indirect_driver.category", "indirect_driver.details"],
    "soil": ["soil.name", "soil.depth"],
    "ecosystem_service": [
        "ecosystem_service.category",
        "ecosystem_service.term",
        "ecosystem_service.details",
    ],
    "management_measure": ["management_measure.description", "management_measure.success"],
    "conservation_area": [
        "conservation_area.name",
        "conservation_area.description",
        "conservation_area.success",
    ],
    "impulse_measure": ["impulse_measure.description", "impulse_measure.success"],
    "location": ["location.country", "location.federal_state", "location.name"],
    "ecosystem_type": [
        "ecosystem_type.category",
        "ecosystem_type.term",
        "ecosystem_type.description",
    ],
}
for p in predictions_flat:
    for c in compounds.keys():
        flattened_lists = flatten(p, c, compounds[c])
        p.update(dict(zip(compounds[c], flattened_lists)))
        if c in p:
            del p[c]

for g in ground_truth_filtered_flat:
    for c in compounds.keys():
        flattened_lists = flatten(g, c, compounds[c])
        g.update(dict(zip(compounds[c], flattened_lists)))
        if c in g:
            del g[c]

In [ ]:
df_predictions = pd.DataFrame.from_dict(predictions_flat)
df_ground_truth = pd.DataFrame.from_dict(ground_truth_filtered_flat)

pd.concat(
    [
        df_ground_truth["id"],
        df_ground_truth["taxa.german_name"],
        df_predictions["taxa.german_name"],
    ],
    axis=1,
)